# PyTorch Basics for LLM Training

**Goal:** Get comfortable with the PyTorch primitives that underpin every LLM training loop.

| Section | Topic |
|---------|-------|
| 1 | Custom Dataset classes (map-style & iterable) |
| 2 | DataLoader with `collate_fn` for dynamic padding |
| 3 | Automatic Mixed Precision (AMP) with memory comparison |
| 4 | `nn.Module` encapsulation, parameter freezing |
| 5 | Exercise: build a mini GPT-like transformer block from scratch |
| 6 | GPU memory monitoring utilities |

In [ ]:
# Install dependencies (run once)
# !pip install torch transformers datasets sentencepiece

## 1. Custom Dataset Classes

Most LLM training uses either **map-style** datasets (indexable, random access) or **iterable** datasets (streaming, no random access). PyTorch supports both.

In [ ]:
import torch
from torch.utils.data import Dataset, IterableDataset, DataLoader
from typing import Iterator
import json

# ------------------------------------------------------------
# 1a. Map-style Dataset -- the most common kind
# ------------------------------------------------------------

class ConversationDataset(Dataset):
    """A map-style dataset of conversations.

    Each item is a list of messages: [{"role": "user", "content": ...}, ...]
    Suitable for instruction-tuning data stored in a JSONL file.
    """
    def __init__(self, path: str, tokenizer=None, max_length: int = 512):
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.samples: list[dict] = []

        if path.endswith(".jsonl"):
            with open(path) as f:
                self.samples = [json.loads(line) for line in f if line.strip()]
        elif path.endswith(".json"):
            with open(path) as f:
                self.samples = json.load(f)
        else:
            raise ValueError(f"Unsupported format: {path}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        sample = self.samples[idx]
        if self.tokenizer is not None:
            # Build the prompt from messages using the chat template
            text = self.tokenizer.apply_chat_template(
                sample["messages"], tokenize=False
            )
            encoding = self.tokenizer(
                text, truncation=True, max_length=self.max_length,
                return_tensors="pt"
            )
            return {
                "input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0),
            }
        return sample


# Create a tiny demo file and test
import tempfile, os

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False)
for i in range(5):
    json.dump({
        "messages": [
            {"role": "user", "content": f"Question {i}"},
            {"role": "assistant", "content": f"Answer {i}"}
        ]
    }, tmp)
    tmp.write("\n")
tmp.close()

ds = ConversationDataset(tmp.name)
print(f"Dataset size: {len(ds)}")
print(f"First item: {ds[0]}")
os.unlink(tmp.name)

In [ ]:
# ------------------------------------------------------------
# 1b. IterableDataset -- for streaming / large-scale data
# ------------------------------------------------------------

class StreamingConversationDataset(IterableDataset):
    """Stream conversations from a JSONL file without loading all into RAM.

    Each worker in a DataLoader gets its own shard automatically.
    """
    def __init__(self, path: str, tokenizer=None, max_length: int = 512):
        self.path = path
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __iter__(self) -> Iterator[dict]:
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None:
            # Shard across workers: each worker reads a subset of lines
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
        else:
            worker_id = 0
            num_workers = 1

        with open(self.path) as f:
            for i, line in enumerate(f):
                if not line.strip():
                    continue
                if i % num_workers != worker_id:
                    continue
                sample = json.loads(line)
                if self.tokenizer is not None:
                    text = self.tokenizer.apply_chat_template(
                        sample["messages"], tokenize=False
                    )
                    yield self.tokenizer(text, truncation=True,
                                        max_length=self.max_length)
                else:
                    yield sample


# Demo with the same file
tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False)
for i in range(10):
    json.dump({"messages": [
        {"role": "user", "content": f"Q{i}"},
        {"role": "assistant", "content": f"A{i}"}
    ]}, tmp)
    tmp.write("\n")
tmp.close()

stream_ds = StreamingConversationDataset(tmp.name)
for i, item in enumerate(stream_ds):
    if i >= 3:
        break
    print(item)
os.unlink(tmp.name)

In [ ]:
# ------------------------------------------------------------
# TODO: Exercise -- Implement a dataset that yields text chunks
# ------------------------------------------------------------
# Create a TextChunkDataset (map-style) that:
#  1. Reads a raw .txt file
#  2. Splits into overlapping chunks of `chunk_size` tokens
#  3. Returns {"input_ids": tensor, "labels": tensor}
#     (labels are the same as input_ids for causal LM training)
#
# class TextChunkDataset(Dataset):
#     def __init__(self, path: str, tokenizer, chunk_size: int = 256, overlap: int = 32):
#         ...  # your code
#     def __len__(self): ...
#     def __getitem__(self, idx): ...

## 2. DataLoader with `collate_fn` for Dynamic Padding

Sequences in a batch have different lengths. A good `collate_fn` pads them to the **longest** sequence in the batch (not a fixed max length), saving compute.

In [ ]:
def collate_for_causal_lm(batch: list[dict]) -> dict[str, torch.Tensor]:
    """Pad a batch of variable-length sequences to the longest in the batch.

    Returns:
        input_ids:      [B, max_len] padded with pad_token_id
        attention_mask:  [B, max_len] 1s for real tokens, 0s for padding
        labels:          [B, max_len] same as input_ids but -100 for padding
    """
    pad_token_id = batch[0].get("pad_token_id", 0)

    # Find the max length in this batch
    max_len = max(item["input_ids"].size(0) for item in batch)

    input_ids_list, attention_mask_list, labels_list = [], [], []

    for item in batch:
        ids = item["input_ids"]
        seq_len = ids.size(0)
        pad_len = max_len - seq_len

        # Pad input_ids
        padded_ids = torch.cat([
            ids,
            torch.full((pad_len,), pad_token_id, dtype=ids.dtype)
        ])
        input_ids_list.append(padded_ids)

        # Build attention mask
        mask = torch.cat([
            torch.ones(seq_len, dtype=torch.long),
            torch.zeros(pad_len, dtype=torch.long)
        ])
        attention_mask_list.append(mask)

        # Labels: -100 on padding so CrossEntropyLoss ignores them
        labels = item.get("labels", ids.clone())
        padded_labels = torch.cat([
            labels,
            torch.full((pad_len,), -100, dtype=labels.dtype)
        ])
        labels_list.append(padded_labels)

    return {
        "input_ids":      torch.stack(input_ids_list),
        "attention_mask":  torch.stack(attention_mask_list),
        "labels":          torch.stack(labels_list),
    }


# Demo -- simulate a batch of different-length sequences
fake_batch = [
    {"input_ids": torch.tensor([1, 2, 3, 4, 5])},
    {"input_ids": torch.tensor([1, 2, 3])},
    {"input_ids": torch.tensor([1, 2, 3, 4, 5, 6, 7, 8])},
]
batched = collate_for_causal_lm(fake_batch)
for k, v in batched.items():
    print(f"{k}: shape={v.shape}\n{v}\n")

In [ ]:
# ------------------------------------------------------------
# DataLoader integration
# ------------------------------------------------------------
class SimpleTokenDataset(Dataset):
    def __init__(self, sequences: list[list[int]]):
        self.data = [torch.tensor(s, dtype=torch.long) for s in sequences]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {"input_ids": self.data[idx]}


token_ds = SimpleTokenDataset([
    [1, 2, 3, 4],
    [1, 2, 3, 4, 5, 6, 7],
    [1, 2, 3],
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    [1, 2, 3, 4, 5],
])

loader = DataLoader(
    token_ds,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_for_causal_lm,
)

for batch_idx, batch in enumerate(loader):
    print(f"Batch {batch_idx}:")
    print(f"  input_ids shape: {batch['input_ids'].shape}")
    print(f"  labels shape:    {batch['labels'].shape}")
    print()

## 3. Automatic Mixed Precision (AMP)

AMP trains with `float16` (forward/backward) while keeping a `float32` master copy of weights. This cuts memory ~40% and speeds up training on modern GPUs with almost no accuracy loss.

In [ ]:
# ------------------------------------------------------------
# Compare memory usage: FP32 vs AMP (FP16+FP32 master)
# ------------------------------------------------------------

def get_gpu_memory_mb() -> dict:
    """Return current GPU memory usage in MB."""
    if not torch.cuda.is_available():
        return {"allocated": 0, "reserved": 0, "free": 0}
    free, total = torch.cuda.mem_get_info()
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved  = torch.cuda.memory_reserved()  / 1024**2
    free_mb   = free / 1024**2
    return {"allocated_MB": allocated, "reserved_MB": reserved, "free_MB": free_mb}


def create_big_tensors(n: int = 4, size: int = 1024) -> list[torch.Tensor]:
    """Create n random (size x size) float tensors to simulate model parameters."""
    return [torch.randn(size, size) for _ in range(n)]


# --- FP32 baseline ---
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    print("=== FP32 ===")
    fp32_tensors = [t.cuda() for t in create_big_tensors(8, 4096)]
    print(f"After allocating FP32 tensors: {get_gpu_memory_mb()}")

    # Free them
    del fp32_tensors
    torch.cuda.empty_cache()

    print("\n=== FP16 ===")
    fp16_tensors = [t.half().cuda() for t in create_big_tensors(8, 4096)]
    print(f"After allocating FP16 tensors: {get_gpu_memory_mb()}")
    print("FP16 uses half the memory of FP32!")

    del fp16_tensors
    torch.cuda.empty_cache()
else:
    print("CUDA not available -- skipping GPU memory demo.")
    print("Run this on a machine with a GPU to see the difference.")

In [ ]:
# ------------------------------------------------------------
# AMP training loop example
# ------------------------------------------------------------

class DummyModel(torch.nn.Module):
    def __init__(self, vocab_size: int = 1000, d_model: int = 256):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab_size, d_model)
        self.ffn   = torch.nn.Sequential(
            torch.nn.Linear(d_model, 4 * d_model),
            torch.nn.GELU(),
            torch.nn.Linear(4 * d_model, d_model),
        )
        self.lm_head = torch.nn.Linear(d_model, vocab_size)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.embed(input_ids)
        x = self.ffn(x)
        return self.lm_head(x)


def train_step_amp(model, optimizer, batch, scaler, device):
    """One training step with AMP + gradient scaling."""
    input_ids = batch["input_ids"].to(device)
    labels    = batch["labels"].to(device)

    with torch.autocast(device_type=device.type, dtype=torch.float16):
        logits = model(input_ids)                      # FP16 forward
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            ignore_index=-100
        )

    optimizer.zero_grad()
    scaler.scale(loss).backward()                       # scaled loss -> FP16 grads
    scaler.unscale_(optimizer)                          # unscale for clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    return loss.item()


# Quick smoke test on CPU (autocast does nothing on CPU but the code runs)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DummyModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

# Fake batch
batch = {
    "input_ids": torch.randint(0, 1000, (2, 64)),
    "labels":    torch.randint(0, 1000, (2, 64)),
}

if device.type == "cuda":
    loss = train_step_amp(model, optimizer, batch, scaler, device)
    print(f"AMP training step completed. Loss = {loss:.4f}")
else:
    print("Running on CPU -- AMP requires CUDA, skipping train_step_amp.")
    print("The code structure is correct and will work on GPU.")

## 4. `nn.Module`: Encapsulation, Parameter Freezing

Every model in PyTorch subclasses `nn.Module`. Key responsibilities: define layers in `__init__`, compute in `forward()`, and manage which parameters get gradients.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

# ------------------------------------------------------------
# A clean nn.Module example: LoRA-style adapter
# ------------------------------------------------------------

class LoraAdapter(nn.Module):
    """A minimal LoRA adapter that wraps a frozen linear layer.

    - The original layer is frozen (requires_grad=False).
    - Only the low-rank A and B matrices are trainable.
    """
    def __init__(self, original: nn.Linear, rank: int = 8, alpha: float = 1.0):
        super().__init__()
        self.original = original
        self.rank = rank
        self.alpha = alpha

        # Freeze the original weights
        for p in self.original.parameters():
            p.requires_grad = False

        # Low-rank decomposition: W_delta = A @ B, where
        # A: [in_features, rank], B: [rank, out_features]
        self.A = nn.Parameter(torch.randn(original.in_features, rank) * 0.02)
        self.B = nn.Parameter(torch.zeros(rank, original.out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        frozen_out = self.original(x)                         # frozen
        lora_out   = (x @ self.A) @ self.B * (self.alpha / self.rank)  # trainable
        return frozen_out + lora_out


# Test
linear = nn.Linear(64, 64)
lora   = LoraAdapter(linear, rank=4)

# Count parameters
total     = sum(p.numel() for p in lora.parameters())
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"Total params: {total}")
print(f"Trainable:    {trainable} ({100*trainable/total:.1f}%)")

# Forward pass works
x = torch.randn(2, 64)
y = lora(x)
print(f"Input: {x.shape} -> Output: {y.shape}")

In [ ]:
# ------------------------------------------------------------
# Parameter freezing utilities
# ------------------------------------------------------------

def freeze_module(module: nn.Module) -> None:
    """Disable gradients for all parameters in a module."""
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_module(module: nn.Module) -> None:
    """Enable gradients for all parameters in a module."""
    for p in module.parameters():
        p.requires_grad = True


def get_trainable_params(model: nn.Module) -> list[str]:
    """Return names of parameters that require gradients."""
    return [n for n, p in model.named_parameters() if p.requires_grad]


def print_param_summary(model: nn.Module):
    """Print a quick summary of trainable vs frozen parameters."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"Trainable: {trainable:,}  |  Frozen: {frozen:,}  |  "
          f"Total: {trainable+frozen:,}")


# Demo: freeze the FFN, keep embedding + lm_head trainable
model2 = DummyModel().to(device)
print("Before freezing:")
print_param_summary(model2)

freeze_module(model2.ffn)
print("\nAfter freezing ffn:")
print_param_summary(model2)
print(f"Trainable param names: {get_trainable_params(model2)}")

## 5. Exercise: Build a Mini GPT-like Transformer Block

Now you will build a simplified transformer decoder block from scratch. This is the core building block of GPT, LLaMA, Qwen, and every other decoder-only LLM.

### Block diagram

```
x -> LayerNorm -> MultiHeadAttention -> Residual(+) -> LayerNorm -> FFN -> Residual(+) -> out
```

### Specifications

- `d_model`: 256
- `n_heads`: 8
- `ffn_multiplier`: 4 (so FFN hidden dim = 1024)
- Use `nn.LayerNorm`, `nn.Linear`, `F.scaled_dot_product_attention`
- The block should handle causal masking automatically

Fill in the `# TODO` sections.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention with causal masking."""

    def __init__(self, d_model: int = 256, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model  = d_model
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.scale    = self.head_dim ** -0.5

        # Combined QKV projection (common optimization)
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape  # batch, seq_len, d_model

        # Project to Q, K, V
        qkv = self.qkv(x)  # [B, T, 3*C]
        q, k, v = qkv.chunk(3, dim=-1)

        # Reshape to multi-head: [B, n_heads, T, head_dim]
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention with causal mask
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=None, is_causal=True, dropout_p=self.dropout.p
        )

        # Merge heads: [B, T, C]
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    """One decoder block: Pre-LN -> Attention -> Residual -> Pre-LN -> FFN -> Residual."""

    def __init__(self, d_model: int = 256, n_heads: int = 8,
                 ffn_multiplier: int = 4, dropout: float = 0.1):
        super().__init__()

        # Pre-LayerNorm (better training stability than post-LN)
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_multiplier * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_multiplier * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Attention sub-layer with residual
        x = x + self.attn(self.ln1(x))
        # FFN sub-layer with residual
        x = x + self.ffn(self.ln2(x))
        return x


# --- Test the block ---
block = TransformerBlock(d_model=256, n_heads=8)
x = torch.randn(2, 32, 256)  # [B=2, T=32, d_model=256]
out = block(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")

# Verify causal masking: output at position i should only depend on positions <= i
# (We test by zeroing the input after position 10 and checking output at position 10)
x2 = torch.randn(2, 32, 256)
x2_modified = x2.clone()
x2_modified[:, 11:, :] = 0.0  # zero out future positions
out_original  = block(x2)
out_modified  = block(x2_modified)
# Positions 0-10 should be identical
assert torch.allclose(out_original[:, :11, :], out_modified[:, :11, :], atol=1e-5), \
    "Causal masking broken!"
print("Causal masking verified -- output[:11] matches when future inputs are zeroed.")

In [ ]:
# ------------------------------------------------------------
# TODO: Build a MiniGPT model using TransformerBlock
# ------------------------------------------------------------
# Assemble:
#   TokenEmbedding -> [TransformerBlock x N] -> LayerNorm -> LM Head
# Then run a forward pass and compute loss.
#
# class MiniGPT(nn.Module):
#     def __init__(self, vocab_size: int = 1000, d_model: int = 256,
#                  n_heads: int = 8, n_layers: int = 4):
#         super().__init__()
#         self.token_embed = nn.Embedding(vocab_size, d_model)
#         self.pos_embed   = nn.Embedding(128, d_model)  # max 128 positions
#         self.blocks = nn.ModuleList([
#             TransformerBlock(d_model, n_heads) for _ in range(n_layers)
#         ])
#         self.ln_final = nn.LayerNorm(d_model)
#         self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)
#
#     def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
#         B, T = input_ids.shape
#         positions = torch.arange(0, T, device=input_ids.device).unsqueeze(0)
#         x = self.token_embed(input_ids) + self.pos_embed(positions)
#         for block in self.blocks:
#             x = block(x)
#         x = self.ln_final(x)
#         return self.lm_head(x)
#
# model = MiniGPT()
# fake_ids = torch.randint(0, 1000, (4, 32))
# logits = model(fake_ids)
# print(f"Logits shape: {logits.shape}")  # [4, 32, 1000]
#
# # Compute loss
# loss = F.cross_entropy(logits.view(-1, 1000), fake_ids.view(-1))
# print(f"Loss: {loss.item():.4f}")


## 6. GPU Memory Monitoring Utilities

OOM errors are the #1 frustration in LLM work. These utilities help you track memory.

In [ ]:
import gc
from contextlib import contextmanager

# ------------------------------------------------------------
# 6a. Peak memory tracker context manager
# ------------------------------------------------------------
@contextmanager
def track_gpu_memory(label: str = ""):
    """Context manager that reports GPU memory used inside the block."""
    if not torch.cuda.is_available():
        print(f"[{label}] CUDA not available")
        yield
        return

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    before = torch.cuda.memory_allocated() / 1024**2

    yield

    after   = torch.cuda.memory_allocated() / 1024**2
    peak    = torch.cuda.max_memory_allocated() / 1024**2
    print(f"[{label}] Before: {before:.1f}MB | After: {after:.1f}MB | "
          f"Delta: {after-before:+.1f}MB | Peak: {peak:.1f}MB")


# Demo
with track_gpu_memory("Allocate 100MB"):
    x = torch.randn(1024, 1024 * 25).cuda() if torch.cuda.is_available() else torch.randn(10, 10)
    # ~100 MB: 1024 * 25600 * 4 bytes = ~100 MB

print("\n(If on CPU, CUDA calls are skipped and memory = 0)")

In [ ]:
# ------------------------------------------------------------
# 6b. Model memory breakdown
# ------------------------------------------------------------

def estimate_model_memory(model: nn.Module, dtype_size: int = 4) -> dict:
    """Estimate memory for model parameters, gradients, and optimizer states.

    Args:
        model: The PyTorch model.
        dtype_size: Bytes per parameter (4 for FP32, 2 for FP16/BF16).

    Returns:
        Dict with memory breakdown in MB.
    """
    param_count     = sum(p.numel() for p in model.parameters())
    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    params_mb      = param_count * dtype_size / 1024**2
    grads_mb       = trainable_count * dtype_size / 1024**2
    # AdamW stores two moment estimates (m and v) per trainable parameter
    opt_states_mb  = trainable_count * dtype_size * 2 / 1024**2

    return {
        "total_params":     param_count,
        "trainable_params": trainable_count,
        "params_MB":        params_mb,
        "grads_MB":         grads_mb,
        "opt_states_MB":    opt_states_mb,
        "total_MB":         params_mb + grads_mb + opt_states_mb + params_mb,
        # The *4 rule of thumb: params + grads + opt_states + activations
        # Usually params * 4 for AdamW in FP32, or params * 2 for AdamW in FP16.
    }


# Try it on our MiniGPT
class MiniGPTFull(nn.Module):
    def __init__(self, vocab_size=1000, d_model=256, n_heads=8, n_layers=4):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed   = nn.Embedding(128, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.token_embed(x) + self.pos_embed(pos)
        for blk in self.blocks:
            h = blk(h)
        return self.lm_head(self.ln_final(h))


mg = MiniGPTFull()
mem = estimate_model_memory(mg, dtype_size=4)
print("FP32 training memory estimate:")
for k, v in mem.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.1f} MB")
    else:
        print(f"  {k}: {v:,}")

print(f"\nBF16 estimate:")
mem_bf16 = estimate_model_memory(mg, dtype_size=2)
print(f"  total_MB: {mem_bf16['total_MB']:.1f} MB")

In [ ]:
# ------------------------------------------------------------
# 6c. OOM-safe forward pass wrapper
# ------------------------------------------------------------

def safe_forward(model: nn.Module, batch: dict, device: torch.device,
                 max_retries: int = 3) -> torch.Tensor | None:
    """Attempt a forward pass; if OOM, clear cache and halve batch size."""
    for attempt in range(max_retries):
        try:
            input_ids = batch["input_ids"].to(device)
            return model(input_ids)
        except torch.cuda.OutOfMemoryError:
            B, T = batch["input_ids"].shape
            new_B = max(1, B // 2)
            print(f"[OOM] attempt {attempt+1}: batch {B}x{T}. Retrying with {new_B}x{T}")
            torch.cuda.empty_cache()
            gc.collect()
            # Truncate batch
            batch["input_ids"] = batch["input_ids"][:new_B]
            if "labels" in batch:
                batch["labels"] = batch["labels"][:new_B]
    print("[OOM] All retries exhausted.")
    return None

print("safe_forward defined. Use this pattern in your training loops.")

---

## What You've Built

| Skill | Applied In |
|-------|-----------|
| Custom `Dataset` / `IterableDataset` | Loading conversation data, text chunking |
| `collate_fn` with dynamic padding | Efficient batching of variable-length sequences |
| AMP with `torch.autocast` + `GradScaler` | Halving GPU memory, faster training |
| `nn.Module` with parameter freezing | LoRA adapters, partial fine-tuning |
| Transformer block from scratch | Understanding GPT internals |
| GPU memory monitoring | Debugging OOM errors, capacity planning |

You are ready to read and modify real LLM training code.